# Módulo 08 - Tratamento de Erros

---

Todo programador erra, e todo programa, cedo ou tarde, encontra dados inesperados. Neste módulo você vai aprender a **reconhecer**, **entender** e **tratar** os três tipos de erro que aparecem em Python: os **erros de sintaxe**, que impedem o código de rodar; os **erros em tempo de execução** (as exceções), que interrompem o programa no meio do caminho; e os **erros de lógica**, os mais traiçoeiros, em que o programa roda sem reclamar, mas entrega o resultado errado. Partindo do `try/except` visto no Módulo 03, vamos aprofundar com `else`, `raise`, a hierarquia de exceções e as exceções personalizadas, tudo aplicado a um problema real: um processamento diário de faturas de uma empresa de telecomunicações que quebra quando os dados chegam fora do padrão. Saber tratar erros é o que separa um *script* que "funciona na minha máquina" de um programa pronto para produção.

Curso: Ready To Deploy

Criado por: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Tópicos

| **Tópico** | Descrição |
| --- | --- |
| 1. Tipos de erros | Visão geral dos três tipos de erro em Python: sintaxe, execução e lógica. |
| 2. Erros de sintaxe | Por que o código nem chega a rodar, os erros de sintaxe mais comuns e como ler a mensagem para corrigi-los. |
| 3. Erros em tempo de execução | Exceções comuns, *traceback*, hierarquia de exceções, `try/except/else/finally`, `raise` e exceções personalizadas, aplicados a um processamento de faturas. |
| 4. Erros de lógica | Como encontrar erros que não geram mensagem nenhuma, usando `print()` e `assert`, incluindo laços infinitos. |

---

## 1. Tipos de erros

Antes de aprender a tratar um erro, é preciso saber **de que tipo** ele é. Cada tipo aparece em um momento diferente e pede uma estratégia diferente para ser resolvido.

### 1.1 Definição

| Tipo | Quando acontece | O Python avisa? | Exemplo | Como resolver |
| --- | --- | --- | --- | --- |
| **Sintaxe** | Antes da execução, quando o Python lê o código | Sim, com `SyntaxError`, e **nenhuma linha** é executada | Esquecer os dois-pontos (`:`) de um `if` | Corrigir o código |
| **Execução** (exceção) | Durante a execução, em uma linha específica | Sim, com uma exceção (`ZeroDivisionError`, `ValueError`...), e o programa para **naquela linha** | Dividir por zero | Corrigir o código ou **tratar** a exceção com `try/except` |
| **Lógica** | Durante a execução | **Não**: o programa roda até o fim | Calcular uma média com a fórmula errada | Conferir resultados e **depurar** o código |

**Erro de sintaxe:** o código está escrito de um jeito que o Python não entende. Por isso a linha abaixo está comentada: se ela fosse executada, a célula inteira falharia antes mesmo de começar.

In [1]:
idade = 19

# if idade >= 18    # ❌ SyntaxError: faltam os dois-pontos no final
if idade >= 18:     # ✅ forma correta
    print('Maior de idade')

Maior de idade


**Erro em tempo de execução:** o código está escrito corretamente, mas uma operação é impossível com os valores recebidos. Aqui usamos o `try/except`, visto no Módulo 03, para mostrar o erro sem travar o notebook.

In [2]:
try:
    print(1 / 0)
except ZeroDivisionError as erro:
    # type(erro).__name__ mostra o nome da exceção
    print(f'{type(erro).__name__}: {erro}')

ZeroDivisionError: division by zero


**Erro de lógica:** o código roda sem nenhuma mensagem, mas o resultado está errado. Qual é a média entre 8 e 10?

In [3]:
nota_1 = 8
nota_2 = 10

media = nota_1 + nota_2 / 2      # ❌ a divisão acontece antes da soma: 8 + 5
print(media)

media = (nota_1 + nota_2) / 2    # ✅ os parênteses garantem a ordem correta
print(media)

13.0
9.0


> ⚠️ **Atenção:** o erro de lógica é o mais perigoso dos três, justamente porque o Python não reclama. Um relatório com um número errado pode passar despercebido e levar a decisões erradas.

---

## 2. Erros de sintaxe

A **sintaxe** é o conjunto de regras de escrita de uma linguagem, como a gramática de um idioma. Quando uma regra é quebrada, o Python não consegue nem entender o que deve fazer, e por isso **não executa nada**.

### 2.1 Definição

Antes de executar uma célula, o Python lê **todo** o código dela para entendê-lo. Se encontrar um erro de sintaxe nessa leitura, ele gera um `SyntaxError` e **nenhuma linha é executada**, nem mesmo as que vêm antes do erro.

Para ver isso acontecer sem travar o notebook, vamos usar duas funções nativas que recebem um código guardado em uma *string*:

- `exec(codigo)`: executa o código;
- `compile(codigo, nome, 'exec')`: apenas **analisa** o código, sem executá-lo.

Elas servem só para esta demonstração. No dia a dia você escreve o código diretamente na célula.

In [4]:
carrinho_compras = [
    {'id': 3184, 'preco': 37.65, 'quantidade': 10},
    {'id': 1203, 'preco': 81.20, 'quantidade': 2},
    {'id': 8921, 'preco': 15.90, 'quantidade': 2},
]

In [5]:
# O print está ANTES do erro de sintaxe (falta ':' no for)...
codigo_com_erro = '''print('Esta mensagem vem antes do erro')
for produto in carrinho_compras
    print(produto)
'''

try:
    exec(codigo_com_erro)
except SyntaxError as erro:
    # ...e mesmo assim ele não é exibido: nada foi executado
    print(f'{type(erro).__name__}: {erro.msg} (linha {erro.lineno})')

SyntaxError: expected ':' (linha 2)


Repare que a mensagem `'Esta mensagem vem antes do erro'` **não apareceu**. Guarde essa diferença: veremos na seção 3 que, com erros de execução, as linhas anteriores ao erro são executadas normalmente.

### 2.2 Erros de sintaxe comuns

Para testar vários exemplos, vamos criar uma função que analisa um código e mostra o erro de sintaxe, indicando a linha e a posição com uma seta (`^`), como o próprio Python faz:

In [6]:
def verificar_sintaxe(codigo: str) -> None:
    '''Analisa o código (sem executá-lo) e exibe o erro de sintaxe, se houver.'''
    codigo = codigo.strip('\n')  # ignora linhas em branco no início e no fim
    try:
        compile(codigo, '<exemplo>', 'exec')
    except SyntaxError as erro:
        linha_com_erro = codigo.splitlines()[erro.lineno - 1]
        print(f'{type(erro).__name__} na linha {erro.lineno}: {erro.msg}')
        print(f'    {linha_com_erro}')
        if erro.offset:
            print('    ' + ' ' * (erro.offset - 1) + '^')
    else:
        print('Nenhum erro de sintaxe encontrado.')

**Exemplo:** esquecer os dois-pontos (`:`) no final de um `for`, `if`, `elif`, `else` ou `def`.

In [7]:
verificar_sintaxe('''
for produto in carrinho_compras
    print(produto)
''')

SyntaxError na linha 1: expected ':'
    for produto in carrinho_compras
                                   ^


**Exemplo:** colocar uma condição no `else`. O `else` significa "em todos os outros casos", então ele **nunca** recebe condição. Para testar outra condição, use `elif`.

In [8]:
verificar_sintaxe('''
for produto in carrinho_compras:
    if produto['id'] == 3184:
        print('Produto 3184')
    else produto['id'] == 1203:
        print('Produto 1203')
''')

SyntaxError na linha 4: expected ':'
        else produto['id'] == 1203:
             ^


In [9]:
# ✅ Forma correta: elif para a segunda condição
for produto in carrinho_compras:
    if produto['id'] == 3184:
        print('Produto 3184')
    elif produto['id'] == 1203:
        print('Produto 1203')

Produto 3184
Produto 1203


**Exemplo:** indentação incorreta. Em Python, os espaços no início da linha definem quais linhas pertencem a um bloco. Esse erro tem até um nome próprio: `IndentationError`.

In [10]:
verificar_sintaxe('''
for produto in carrinho_compras:
print(produto)
''')

IndentationError na linha 2: expected an indented block after 'for' statement on line 1
    print(produto)
    ^


**Exemplo:** esquecer de fechar parênteses, colchetes ou aspas.

In [11]:
verificar_sintaxe('''
print('Total de produtos:', len(carrinho_compras)
''')

SyntaxError na linha 1: '(' was never closed
    print('Total de produtos:', len(carrinho_compras)
         ^


In [12]:
verificar_sintaxe('''
mensagem = 'Bem-vindo ao Ready To Deploy
''')

SyntaxError na linha 1: unterminated string literal (detected at line 1)
    mensagem = 'Bem-vindo ao Ready To Deploy
               ^


**Exemplo:** usar `=` (atribuição) onde deveria ser `==` (comparação).

In [13]:
verificar_sintaxe('''
if len(carrinho_compras) = 3:
    print('Carrinho com 3 produtos')
''')

SyntaxError na linha 1: cannot assign to function call here. Maybe you meant '==' instead of '='?
    if len(carrinho_compras) = 3:
       ^


**Exemplo:** usar `return` fora de uma função. O `return` só faz sentido dentro de um `def`.

In [14]:
verificar_sintaxe('''
idade = 19
if idade > 18:
    return True
''')

SyntaxError na linha 3: 'return' outside function
        return True
        ^


Resumindo os casos mais frequentes:

| Erro | Exemplo errado | Correção |
| --- | --- | --- |
| Faltam os dois-pontos | `for produto in carrinho` | `for produto in carrinho:` |
| Condição no `else` | `else preco > 10:` | `elif preco > 10:` |
| Indentação incorreta | bloco do `for` sem recuo | recuar o bloco com 4 espaços |
| Parêntese ou aspas sem fechar | `print('Olá'` | `print('Olá')` |
| `=` no lugar de `==` | `if total = 3:` | `if total == 3:` |
| `return` fora de função | `return True` solto no código | usar `return` só dentro de um `def` |

### 2.3 Como corrigir

Erros de sintaxe **não podem ser tratados** com `try/except` dentro da própria célula: como o Python não consegue nem ler o código, o `try` nunca chega a ser executado. A única saída é **corrigir o código**. Para isso, leia a mensagem de erro com atenção:

1. **Tipo do erro:** `SyntaxError` ou `IndentationError`;
2. **Linha:** onde o Python percebeu o problema;
3. **Seta `^`:** a posição aproximada do erro na linha;
4. **Descrição:** nas versões mais recentes do Python ela é bem específica, como `expected ':'` ("esperava `:`").

> 💡 **Dica:** o Python aponta o lugar onde **percebeu** o erro, que nem sempre é onde ele **está**. Um parêntese esquecido numa linha costuma ser acusado só na linha seguinte. Se a linha indicada parecer correta, confira também a linha de cima.

> 💡 **Dica:** editores como o Colab e o VS Code sublinham em vermelho boa parte dos erros de sintaxe enquanto você digita. Fique de olho nesses avisos antes de executar a célula.

---

## 3. Erros em tempo de execução

Um código com a sintaxe perfeita ainda pode falhar: basta receber um dado inesperado. Esses erros, chamados de **exceções**, são os mais comuns no dia a dia, e é aqui que o tratamento de erros faz mais diferença.

### 3.1 Motivação

Você trabalha como analista de dados em uma empresa de telecomunicações e precisa informar ao time de vendas **quanto a empresa vai receber este mês**. Todos os dias, o time de engenharia envia um arquivo CSV com as faturas dos clientes:

| Coluna | Significado |
| --- | --- |
| `customerID` | Identificador do cliente |
| `PaymentMethod` | Forma de pagamento |
| `MonthlyCharges` | Valor da fatura mensal |
| `TotalCharges` | Valor total já pago pelo cliente |
| `Churn` | O cliente cancelou o serviço? |

Primeiro, vamos criar os arquivos de exemplo. O arquivo do **dia 1** chegou no formato combinado:

In [15]:
def criar_arquivo(nome_arquivo: str, conteudo: str) -> None:
    '''Cria (ou sobrescreve) um arquivo de texto com o conteúdo informado.'''
    with open(nome_arquivo, mode='w', encoding='utf-8') as arquivo:
        arquivo.write(conteudo)


criar_arquivo('telecom_dia_01.csv', '''customerID,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7010-BRBUU,Credit card (automatic),24.1,1734.65,No
9688-YGXVR,Credit card (automatic),88.15,3973.2,No
9286-DOJGF,Bank transfer (automatic),74.95,2869.85,Yes
6994-KERXL,Electronic check,55.9,238.5,No
2181-UAESM,Electronic check,53.45,119.5,No
4312-GVYNH,Bank transfer (automatic),49.85,3370.2,No
2495-KZNFB,Electronic check,90.65,2989.6,No
4367-NHWMM,Mailed check,24.9,24.9,No
8898-KASCD,Mailed check,35.55,1309.15,No
''')

Esta é a função que você escreveu para somar as faturas. Ela lê o arquivo linha a linha e pega o valor da **3ª coluna** (`MonthlyCharges`, índice `2`):

In [16]:
def somar_faturas(nome_arquivo: str) -> float:
    faturas = []

    with open(nome_arquivo, mode='r', encoding='utf-8') as arquivo:
        arquivo.readline()  # descarta o cabeçalho
        for linha in arquivo:
            colunas = linha.strip().split(',')
            fatura = float(colunas[2])  # 3ª coluna: MonthlyCharges
            faturas.append(fatura)

    return sum(faturas)


total_a_receber = somar_faturas('telecom_dia_01.csv')
print(f'Total a receber: {total_a_receber:.2f}')

Total a receber: 497.50


Tudo certo no dia 1. Mas no **dia 2**, a engenharia mudou a ordem das colunas sem avisar: `MonthlyCharges` agora é a 2ª coluna, e a 3ª passou a ser `PaymentMethod`.

In [17]:
criar_arquivo('telecom_dia_02.csv', '''customerID,MonthlyCharges,PaymentMethod,TotalCharges,Churn
7010-BRBUU,24.1,Credit card (automatic),1734.65,No
9688-YGXVR,88.15,Credit card (automatic),3973.2,No
9286-DOJGF,74.95,Bank transfer (automatic),2869.85,Yes
6994-KERXL,55.9,Electronic check,238.5,No
2181-UAESM,53.45,Electronic check,119.5,No
4312-GVYNH,49.85,Bank transfer (automatic),3370.2,No
2495-KZNFB,90.65,Electronic check,2989.6,No
4367-NHWMM,24.9,Mailed check,24.9,No
8898-KASCD,35.55,Mailed check,1309.15,No
''')

Ao processar o novo arquivo, o programa **quebra**. Sem o `try/except` abaixo (que usamos só para o notebook não travar), a execução pararia aqui com um `ValueError`:

In [18]:
try:
    total_a_receber = somar_faturas('telecom_dia_02.csv')
    print(f'Total a receber: {total_a_receber:.2f}')
except ValueError as erro:
    print(f'{type(erro).__name__}: {erro}')

ValueError: could not convert string to float: 'Credit card (automatic)'


O Python tentou converter o texto `'Credit card (automatic)'` em número, o que é impossível.

**Como podemos fazer o processamento lidar com esse tipo de problema, avisando claramente o que deu errado ou, melhor ainda, continuando a funcionar mesmo com as colunas trocadas?**

### 3.2 Definição

Um **erro em tempo de execução** acontece enquanto o programa está rodando. O código é executado normalmente **até a linha do erro**. Nesse momento, o Python "lança" (ou "levanta") uma **exceção** e, se ninguém a tratar, o programa é interrompido.

Veja a diferença em relação ao erro de sintaxe: aqui, a mensagem antes do erro **é** exibida.

In [19]:
try:
    print('Esta mensagem vem antes do erro')  # é executada
    print(10 / 0)                             # aqui a exceção é lançada
    print('Esta mensagem vem depois do erro') # nunca é executada
except ZeroDivisionError as erro:
    print(f'{type(erro).__name__}: {erro}')

Esta mensagem vem antes do erro
ZeroDivisionError: division by zero


Cada tipo de problema gera uma exceção com um nome diferente. Estas são as mais comuns:

| Exceção | Quando acontece | Exemplo |
| --- | --- | --- |
| `ZeroDivisionError` | Divisão por zero | `10 / 0` |
| `TypeError` | Operação com tipos incompatíveis | `'idade: ' + 30` |
| `ValueError` | Tipo certo, mas valor impossível de usar | `float('abc')` |
| `IndexError` | Posição inexistente em uma lista | `[1, 2, 3][5]` |
| `KeyError` | Chave inexistente em um dicionário | `{'a': 1}['b']` |
| `NameError` | Variável ou função que não existe | `print(variavel_inexistente)` |
| `AttributeError` | Método ou atributo que o objeto não tem | `'texto'.somar()` |
| `FileNotFoundError` | Arquivo que não existe | `open('nao_existe.csv')` |

**Exemplo:** operação numérica impossível, ao dividir uma conta entre zero pessoas.

In [20]:
valor_conta = 132.85
quantidade_pessoas = 0

try:
    valor_por_pessoa = valor_conta / quantidade_pessoas
except ZeroDivisionError as erro:
    print(f'{type(erro).__name__}: {erro}')

ZeroDivisionError: float division by zero


**Exemplo:** combinação de tipos incompatíveis, ao concatenar texto com número.

In [21]:
nome = 'André Perez'
idade = 30

try:
    apresentacao = 'Meu nome é ' + nome + ' e eu tenho ' + idade + ' anos.'
except TypeError as erro:
    print(f'{type(erro).__name__}: {erro}')

# ✅ Com f-string, a conversão para texto é automática
print(f'Meu nome é {nome} e eu tenho {idade} anos.')

TypeError: can only concatenate str (not "int") to str
Meu nome é André Perez e eu tenho 30 anos.


**Exemplo:** acessando uma posição que não existe em uma lista.

In [22]:
anos = [2019, 2020, 2021]

try:
    ano_atual = anos[3]  # as posições válidas são 0, 1 e 2
except IndexError as erro:
    print(f'{type(erro).__name__}: {erro}')

IndexError: list index out of range


**Exemplo:** acessando uma chave que não existe em um dicionário.

In [23]:
cursos = {
    'python': {'nome': 'Python para Análise de Dados', 'duracao_meses': 2.5},
    'sql': {'nome': 'SQL para Análise de Dados', 'duracao_meses': 2},
}

print(cursos['python'])
print(cursos['sql'])

try:
    print(cursos['analista'])
except KeyError as erro:
    print(f'{type(erro).__name__}: {erro}')

{'nome': 'Python para Análise de Dados', 'duracao_meses': 2.5}
{'nome': 'SQL para Análise de Dados', 'duracao_meses': 2}
KeyError: 'analista'


> 💡 **Dica:** muitas vezes dá para **evitar** a exceção em vez de tratá-la. Para dicionários, o método `.get(chave, valor_padrao)` devolve o valor padrão quando a chave não existe, sem gerar erro.

In [24]:
curso = cursos.get('analista', 'Curso não encontrado')
print(curso)

Curso não encontrado


### 3.3 Lendo o *traceback*

Quando uma exceção não é tratada, o Python exibe o ***traceback***: o "rastro" das chamadas de função que levaram até o erro. Ele parece assustador, mas segue sempre a mesma estrutura. O módulo `traceback` permite exibi-lo mesmo quando capturamos a exceção:

In [25]:
import traceback

def calcular_valor_por_pessoa(valor_total: float, quantidade_pessoas: int) -> float:
    return valor_total / quantidade_pessoas

def fechar_conta(valor_total: float, quantidade_pessoas: int) -> None:
    valor = calcular_valor_por_pessoa(valor_total, quantidade_pessoas)
    print(f'Cada pessoa paga {valor:.2f}')

try:
    fechar_conta(132.85, 0)
except ZeroDivisionError:
    traceback.print_exc()  # exibe o traceback completo, como se o erro não fosse tratado

Traceback (most recent call last):
  File "C:\Users\schit\AppData\Local\Temp\ipykernel_37644\1307365568.py", line 11, in <module>
    fechar_conta(132.85, 0)
  File "C:\Users\schit\AppData\Local\Temp\ipykernel_37644\1307365568.py", line 7, in fechar_conta
    valor = calcular_valor_por_pessoa(valor_total, quantidade_pessoas)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\schit\AppData\Local\Temp\ipykernel_37644\1307365568.py", line 4, in calcular_valor_por_pessoa
    return valor_total / quantidade_pessoas
           ~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
ZeroDivisionError: float division by zero


Leia o *traceback* **de baixo para cima**:

1. **Última linha:** o tipo da exceção e a mensagem (`ZeroDivisionError: division by zero`). Comece sempre por aqui;
2. **Linhas logo acima:** o arquivo, o número da linha e a função onde o erro aconteceu (`calcular_valor_por_pessoa`);
3. **Linhas mais acima:** o caminho de chamadas até ali. Quem chamou quem (`fechar_conta` chamou `calcular_valor_por_pessoa`).

> 💡 **Dica:** ao pesquisar um erro na internet, copie a **última linha** do *traceback*. Ela é a parte mais útil e costuma levar direto a explicações e soluções.

### 3.4 Hierarquia de exceções

No Módulo 06 vimos que classes podem **herdar** de outras classes. As exceções do Python são justamente classes organizadas em uma hierarquia de herança. Um trecho simplificado:

```
Exception
 ├── ArithmeticError
 │    └── ZeroDivisionError
 ├── LookupError
 │    ├── IndexError
 │    └── KeyError
 ├── ValueError
 ├── TypeError
 └── OSError
      └── FileNotFoundError
```

Um `except` captura a exceção indicada **e todas as que herdam dela**. A função nativa `issubclass()` confirma essas relações:

In [26]:
print(issubclass(ZeroDivisionError, ArithmeticError))  # True
print(issubclass(IndexError, LookupError))             # True
print(issubclass(KeyError, LookupError))               # True
print(issubclass(ValueError, Exception))               # True: quase tudo herda de Exception

True
True
True
True


**Exemplo:** como `IndexError` e `KeyError` herdam de `LookupError`, um único `except LookupError` trata os dois casos.

In [27]:
def buscar(colecao, chave_ou_posicao):
    try:
        return colecao[chave_ou_posicao]
    except LookupError as erro:
        print(f'Não encontrado ({type(erro).__name__}): {erro}')
        return None

buscar(anos, 10)             # lista: gera IndexError
buscar(cursos, 'analista')   # dicionário: gera KeyError

Não encontrado (IndexError): list index out of range
Não encontrado (KeyError): 'analista'


> ⚠️ **Atenção:** a **ordem dos `except` importa**. O Python usa o **primeiro** que combinar com a exceção. Por isso, coloque sempre as exceções **mais específicas primeiro** e as mais genéricas (como `Exception`) por último. Um `except Exception` no topo capturaria tudo, e os `except` de baixo nunca seriam usados.

> ⚠️ **Atenção:** evite capturar `Exception` (ou usar um `except:` sem nada) só para "sumir com o erro". Isso esconde problemas reais, inclusive erros de digitação no seu próprio código. Capture apenas as exceções que você sabe tratar.

### 3.5 try / except / else / finally

No Módulo 03 vimos o `try`, o `except` e o `finally`. A estrutura completa tem ainda o bloco `else`:

```python
try:
    # código que pode gerar uma exceção
except TipoDeExcecao as erro:
    # executado SE a exceção acontecer
else:
    # executado SE NENHUMA exceção acontecer
finally:
    # executado SEMPRE, com ou sem exceção
```

| Bloco | Obrigatório? | Quando é executado |
| --- | --- | --- |
| `try` | Sim | Sempre, até a primeira exceção |
| `except` | Pelo menos um `except` ou um `finally` | Somente se ocorrer a exceção indicada |
| `else` | Não | Somente se **nenhuma** exceção ocorrer no `try` |
| `finally` | Não | Sempre, no final, aconteça o que acontecer |

**Exemplo:** consultando um ano em uma coleção.

In [28]:
def consultar_ano(anos, posicao: int) -> None:
    try:
        ano = anos[posicao]
    except IndexError:
        print(f'Posição inválida. Escolha um valor entre 0 e {len(anos) - 1}.')
    except TypeError as erro:
        print(f'Coleção não suportada: {erro}')
    else:
        print(f'Ano encontrado: {ano}')
    finally:
        print('Consulta finalizada.')
        print('-' * 30)

In [29]:
lista_anos = [2019, 2020, 2021]
conjunto_anos = {2019, 2020, 2021}

consultar_ano(lista_anos, 1)      # sem erro: executa o else
consultar_ano(lista_anos, 3)      # IndexError
consultar_ano(conjunto_anos, 0)   # TypeError: conjuntos não têm posição

Ano encontrado: 2020
Consulta finalizada.
------------------------------
Posição inválida. Escolha um valor entre 0 e 2.
Consulta finalizada.
------------------------------
Coleção não suportada: 'set' object is not subscriptable
Consulta finalizada.
------------------------------


> 💡 **Dica:** por que usar o `else` em vez de colocar tudo dentro do `try`? Para deixar o `try` com o **mínimo** de código possível: apenas a linha que pode falhar. Assim, uma exceção inesperada em outra parte do código não é capturada por engano pelo `except`.

### 3.6 Lançando exceções com `raise`

Até agora apenas **reagimos** às exceções do Python. Com a palavra-chave `raise`, nós mesmos podemos **lançar** uma exceção quando detectamos uma situação inválida, interrompendo a função e avisando quem a chamou.

**Exemplo:** validando os dados antes de dividir uma conta.

In [30]:
def dividir_conta(valor_total: float, quantidade_pessoas: int) -> float:
    if quantidade_pessoas <= 0:
        raise ValueError(f'A quantidade de pessoas deve ser maior que zero (recebido: {quantidade_pessoas}).')
    return valor_total / quantidade_pessoas

In [31]:
print(dividir_conta(132.85, 5))

try:
    print(dividir_conta(132.85, -2))
except ValueError as erro:
    print(f'Não foi possível dividir a conta: {erro}')

26.57
Não foi possível dividir a conta: A quantidade de pessoas deve ser maior que zero (recebido: -2).


> 💡 **Dica:** sem a validação, `dividir_conta(132.85, -2)` retornaria um valor negativo sem reclamar, um **erro de lógica**. Lançar a exceção transforma um problema silencioso em um erro claro e fácil de encontrar.

Dentro de um `except`, podemos também **repassar** a exceção para frente com um `raise` sozinho. Isso é útil quando a função quer registrar o problema, mas deixar a decisão sobre o que fazer para quem a chamou:

In [32]:
def converter_valor(texto: str) -> float:
    try:
        return float(texto)
    except ValueError:
        print(f"[registro] valor inválido recebido: '{texto}'")
        raise  # repassa a MESMA exceção para quem chamou


try:
    converter_valor('Credit card (automatic)')
except ValueError as erro:
    print(f'Quem chamou recebeu o erro: {erro}')

[registro] valor inválido recebido: 'Credit card (automatic)'
Quem chamou recebeu o erro: could not convert string to float: 'Credit card (automatic)'


Por fim, podemos **trocar** a exceção por outra mais clara para o contexto, mantendo a original como causa, com `raise NovaExcecao(...) from erro`:

In [33]:
def obter_ano_atual(anos: list) -> int:
    try:
        return anos[3]
    except IndexError as erro:
        raise ValueError(f'A lista de anos precisa ter pelo menos 4 elementos (tem {len(anos)}).') from erro


try:
    obter_ano_atual([2019, 2020, 2021])
except ValueError as erro:
    print(f'Erro:  {erro}')
    print(f'Causa: {type(erro.__cause__).__name__}: {erro.__cause__}')  # a exceção original

Erro:  A lista de anos precisa ter pelo menos 4 elementos (tem 3).
Causa: IndexError: list index out of range


> 💡 **Dica:** o `from erro` preserva a exceção original no *traceback*, que passa a mostrar as duas: "a exceção acima foi a causa direta da exceção abaixo". Isso facilita muito a investigação do problema.

### 3.7 Exceções personalizadas

Como as exceções são classes, podemos criar as **nossas**, herdando de `Exception` (ou de uma exceção mais específica). Uma exceção com nome próprio deixa claro **o que** deu errado e permite que quem chama a função trate aquele caso separadamente dos demais.

In [34]:
class FaturaInvalidaError(Exception):
    '''Lançada quando um arquivo de faturas não pode ser processado.'''


try:
    raise FaturaInvalidaError('Linha 5: valor da fatura em branco.')
except FaturaInvalidaError as erro:
    print(f'{type(erro).__name__}: {erro}')

FaturaInvalidaError: Linha 5: valor da fatura em branco.


> 💡 **Dica:** por convenção, o nome de uma exceção termina com `Error`, como as exceções nativas (`ValueError`, `KeyError`). A *docstring* sozinha já basta como corpo da classe, sem precisar de `pass`.

### 3.8 Revisitando a motivação

Vamos voltar ao processamento das faturas. Podemos melhorá-lo em duas frentes:

1. **Robustez:** em vez de confiar na posição fixa da coluna, localizamos a coluna `MonthlyCharges` **pelo nome**, lendo o cabeçalho. Assim, trocar a ordem das colunas deixa de ser um problema;
2. **Mensagens claras:** se ainda assim um dado for inválido (a coluna não existe, um valor está em branco...), lançamos uma `FaturaInvalidaError` dizendo **em qual linha** e **por quê**, preservando a exceção original com `from`.

A função nativa `enumerate()` percorre as linhas do arquivo e, ao mesmo tempo, conta cada uma delas. Com `start=2`, a contagem começa em 2, porque a linha 1 é o cabeçalho.

In [35]:
def somar_faturas(nome_arquivo: str, coluna: str = 'MonthlyCharges') -> float:
    '''Soma os valores da coluna indicada. Lança FaturaInvalidaError se algum dado for inválido.'''
    faturas = []

    with open(nome_arquivo, mode='r', encoding='utf-8') as arquivo:
        cabecalho = arquivo.readline().strip().split(',')

        # 1. Localiza a coluna pelo nome (list.index lança ValueError se não existir)
        try:
            posicao_coluna = cabecalho.index(coluna)
        except ValueError as erro:
            raise FaturaInvalidaError(f"Coluna '{coluna}' não encontrada no cabeçalho.") from erro

        # 2. Converte o valor de cada linha, informando a linha exata em caso de erro
        for numero_linha, linha in enumerate(arquivo, start=2):
            valor = linha.strip().split(',')[posicao_coluna]
            try:
                fatura = float(valor)
            except ValueError as erro:
                raise FaturaInvalidaError(f"Linha {numero_linha}: valor inválido '{valor}'.") from erro
            else:
                faturas.append(fatura)

    return sum(faturas)

Agora os dois arquivos são processados corretamente, e dão o mesmo total, já que só a ordem das colunas mudou:

In [36]:
for nome_arquivo in ['telecom_dia_01.csv', 'telecom_dia_02.csv']:
    total_a_receber = somar_faturas(nome_arquivo)
    print(f'{nome_arquivo}: total a receber = {total_a_receber:.2f}')

telecom_dia_01.csv: total a receber = 497.50
telecom_dia_02.csv: total a receber = 497.50


E se, no **dia 3**, um cliente vier com a fatura em branco? A função não tem como adivinhar o valor correto, então ela **não deve** esconder o problema. Em vez disso, lança um erro claro. Quem chama a função decide o que fazer: aqui, avisamos o time de engenharia em vez de enviar um total errado para o time de vendas.

In [37]:
criar_arquivo('telecom_dia_03.csv', '''customerID,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7010-BRBUU,Credit card (automatic),24.1,1734.65,No
9688-YGXVR,Credit card (automatic),88.15,3973.2,No
9286-DOJGF,Bank transfer (automatic),,2869.85,Yes
6994-KERXL,Electronic check,55.9,238.5,No
''')

In [38]:
for nome_arquivo in ['telecom_dia_01.csv', 'telecom_dia_02.csv', 'telecom_dia_03.csv']:
    try:
        total_a_receber = somar_faturas(nome_arquivo)
    except FaturaInvalidaError as erro:
        print(f'❌ {nome_arquivo}: arquivo rejeitado. {erro} Avise o time de engenharia.')
    except FileNotFoundError:
        print(f'❌ {nome_arquivo}: arquivo não encontrado.')
    else:
        print(f'✅ {nome_arquivo}: total a receber = {total_a_receber:.2f}')

✅ telecom_dia_01.csv: total a receber = 497.50
✅ telecom_dia_02.csv: total a receber = 497.50
❌ telecom_dia_03.csv: arquivo rejeitado. Linha 4: valor inválido ''. Avise o time de engenharia.


Compare com a versão inicial: antes, uma mudança no arquivo derrubava o programa com uma mensagem genérica (`could not convert string to float`). Agora, a troca de colunas é resolvida automaticamente, e um dado realmente inválido gera uma mensagem que diz **o arquivo**, **a linha** e **o valor** com problema. Enquanto isso, os outros arquivos continuam sendo processados.

> 💡 **Dica:** no Módulo 05 somamos listas com `reduce()`. Aqui usamos a função nativa `sum()`, que faz o mesmo de forma mais direta e legível.

---

## 4. Erros de lógica

Um erro de lógica não gera nenhuma exceção: o programa roda até o fim e entrega um resultado, só que **errado**. Como o Python não aponta o problema, cabe a nós desconfiar dos resultados e investigar o código.

### 4.1 Definição

Um **erro de lógica** acontece quando o código faz exatamente o que foi escrito, mas o que foi escrito não é o que queríamos. As causas mais comuns são:

- Fórmulas erradas ou com a ordem das operações trocada (como a média da seção 1);
- Limites errados em laços e fatias, pulando ou repetindo elementos;
- Condições invertidas (`>` no lugar de `<`, `and` no lugar de `or`);
- Laços que nunca terminam (**laços infinitos**).

A principal ferramenta para encontrá-los é a **depuração**: exibir resultados intermediários para descobrir em que ponto o valor calculado deixa de ser o esperado.

### 4.2 Limites de coleções

**Exemplo:** calculando o valor total do carrinho de compras. O resultado esperado é **570,70**: $37{,}65 \times 10 + 81{,}20 \times 2 + 15{,}90 \times 2$.

In [39]:
valor_total = 0

for indice in range(1, len(carrinho_compras)):
    produto = carrinho_compras[indice]
    valor_total += produto['preco'] * produto['quantidade']

print(f'Valor total: {valor_total:.2f}')

Valor total: 194.20


O código rodou sem nenhum erro, mas o valor está muito abaixo do esperado. Algo está errado na **lógica**.

### 4.3 Depurando com `print()`

A forma mais simples de depurar é colocar um `print()` dentro do laço para **ver** o que está acontecendo a cada passo:

In [40]:
valor_total = 0

for indice in range(1, len(carrinho_compras)):
    produto = carrinho_compras[indice]
    print(f'[depuração] índice {indice}: produto {produto["id"]}')  # print temporário
    valor_total += produto['preco'] * produto['quantidade']

print(f'Valor total: {valor_total:.2f}')

[depuração] índice 1: produto 1203
[depuração] índice 2: produto 8921
Valor total: 194.20


Pronto, o problema apareceu: o laço começa no índice `1` e **pula o primeiro produto** (índice `0`, o `3184`). O `range(1, ...)` deveria ser `range(0, ...)`. Melhor ainda: percorrendo a lista diretamente com `for produto in carrinho_compras`, não há índice para errar:

In [41]:
valor_total = 0

for produto in carrinho_compras:
    valor_total += produto['preco'] * produto['quantidade']

print(f'Valor total: {valor_total:.2f}')

Valor total: 570.70


> 💡 **Dica:** depois de encontrar o erro, **remova** os `print()` de depuração, para que não poluam a saída do programa.

> 💡 **Dica:** para se proteger de erros de lógica, você pode verificar se um resultado faz sentido com `assert condição, 'mensagem'`. Se a condição for falsa, o Python lança um `AssertionError` com a mensagem. Se for verdadeira, nada acontece.

In [42]:
quantidade_de_produtos = len(carrinho_compras)
produtos_somados = 0

for produto in carrinho_compras:
    produtos_somados += 1

# Se algum produto tivesse sido pulado, esta linha lançaria um AssertionError
assert produtos_somados == quantidade_de_produtos, 'Algum produto não foi somado!'
print('Conferência OK: todos os produtos foram somados.')

Conferência OK: todos os produtos foram somados.


### 4.4 Laços infinitos

Um tipo especial de erro de lógica é o **laço infinito**: um laço cuja condição de parada nunca é atingida, deixando o programa "preso" para sempre.

Ele é mais comum com o laço `while` ("enquanto"), que repete um bloco **enquanto** uma condição for verdadeira. Diferente do `for`, que percorre uma coleção com fim definido, o `while` depende de que algo dentro dele **mude a condição** em algum momento.

In [43]:
tentativa = 1

while tentativa <= 3:
    print(f'Tentativa de conexão {tentativa}...')
    tentativa += 1  # sem esta linha, tentativa seria sempre 1 e o laço nunca terminaria

print('Fim das tentativas.')

Tentativa de conexão 1...
Tentativa de conexão 2...
Tentativa de conexão 3...
Fim das tentativas.


Se a linha `tentativa += 1` fosse esquecida, a condição `tentativa <= 3` seria **sempre** verdadeira e o laço rodaria para sempre. Por isso ela é tão importante. Outra forma de garantir a saída é usar o `break`, visto no Módulo 03, para interromper o laço quando uma condição for atingida:

In [44]:
contador = 0

while True:          # a condição é sempre verdadeira...
    contador += 1
    if contador > 10:
        break        # ...então o break é a ÚNICA saída do laço

print(f'O laço executou {contador - 1} vezes.')

O laço executou 10 vezes.


> ⚠️ **Atenção:** se uma célula ficar executando por muito mais tempo que o esperado, desconfie de um laço infinito. No Colab, interrompa a execução clicando no botão de **parar** ao lado da célula (ou com o atalho `Ctrl + M` + `I`). Um laço infinito que acumula dados na memória pode até derrubar a sessão inteira.

---

## Resumo do Módulo

| Tipo de erro | Quando acontece | Sinal | Como lidar |
| --- | --- | --- | --- |
| Sintaxe | Antes da execução | `SyntaxError` / `IndentationError`, nada é executado | Ler a mensagem (linha e `^`) e corrigir o código |
| Execução | Durante a execução | Exceção (`ValueError`, `KeyError`...) e *traceback* | Corrigir, evitar (validar dados, `.get()`) ou tratar com `try/except` |
| Lógica | Durante a execução | Nenhum: só o resultado errado | Conferir resultados, depurar com `print()` e usar `assert` |

| Recurso | Para que serve |
| --- | --- |
| `try` / `except` | Captura e trata uma exceção |
| `else` | Executa somente se não houve exceção no `try` |
| `finally` | Executa sempre, com ou sem exceção |
| `raise Excecao('mensagem')` | Lança uma exceção |
| `raise` (sozinho, dentro do `except`) | Repassa a exceção atual para quem chamou |
| `raise NovaExcecao(...) from erro` | Troca a exceção, preservando a original como causa |
| `class MeuErro(Exception):` | Cria uma exceção personalizada |
| `traceback.print_exc()` | Exibe o *traceback* de uma exceção capturada |
| `assert condição, 'mensagem'` | Confere se um resultado faz sentido |

Boas práticas: mantenha o `try` pequeno, capture exceções **específicas** (das mais específicas para as mais genéricas), nunca "engula" erros sem tratá-los e prefira mensagens que digam **onde** e **por que** algo falhou.